# Chateau Combo - IA Extreme (Reinforcement Learning)

Entrainement par renforcement avec self-play sur **A100**.

**Architecture:**
- MaskablePPO avec masquage d'actions invalides
- Transformer (4 layers, 8 heads, dim 128)
- Self-play avec pool d'adversaires

**Estimation:** ~20M timesteps en 2-4 heures sur A100

## 1. Verifier le GPU

**IMPORTANT:** Verifier que tu as bien un A100 :
- Runtime > Change runtime type > A100 GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoire: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Verifier que c'est bien un A100
    gpu_name = torch.cuda.get_device_name(0)
    if "A100" in gpu_name:
        print("\n[OK] A100 detecte !")
    else:
        print(f"\n[ATTENTION] GPU detecte: {gpu_name}")
        print("Va dans Runtime > Change runtime type > A100 GPU")

## 2. Monter Google Drive

Les checkpoints seront sauvegardes sur Drive pour ne pas les perdre si Colab se deconnecte.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Creer les repertoires
!mkdir -p /content/drive/MyDrive/chato_rl/checkpoints
!mkdir -p /content/drive/MyDrive/chato_rl/logs
print("[OK] Drive monte et repertoires crees")

## 3. Cloner le repository et installer

In [ ]:
# Cloner le repo (remplace par ton URL)
!git clone https://github.com/TON_USERNAME/chato-combourg.git
%cd chato-combourg/rl

# Installer les dependances
!pip install -e . -q

print("\n[OK] Installation terminee")

## 4. Verifier que tout fonctionne

In [ ]:
from chato_rl.game import CardDatabase, GameEngine
from chato_rl.env import ChatoEnv

# Test rapide
db = CardDatabase()
print(f"[OK] {db.num_cards} cartes chargees")

env = ChatoEnv(num_players=2)
obs, _ = env.reset()
print(f"[OK] Environnement cree - {int(obs['action_mask'].sum())} actions valides")

print("\nTout est pret !")

## 5. Configurer l'entrainement

In [ ]:
from chato_rl.training.config import A100Config

# Configuration A100 optimisee
config = A100Config()

# Ajuster le nombre de timesteps si besoin
# config.total_timesteps = 10_000_000  # 10M pour un premier test

print("Configuration A100:")
print(f"  - Timesteps: {config.total_timesteps:,}")
print(f"  - Envs paralleles: {config.num_envs}")
print(f"  - Batch size: {config.batch_size}")
print(f"  - Transformer: d={config.d_model}, heads={config.num_heads}, layers={config.num_layers}")
print(f"  - Features dim: {config.features_dim}")
print(f"  - Device: {config.device}")

## 6. Lancer TensorBoard (optionnel)

Pour suivre la progression en temps reel.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/chato_rl/logs

## 7. Lancer l'entrainement

**Estimation pour 20M timesteps sur A100:** 2-4 heures

Les checkpoints sont sauvegardes sur Google Drive toutes les 100k steps.

In [ ]:
from chato_rl.training import SelfPlayTrainer
import warnings
warnings.filterwarnings('ignore')

# Creer le trainer
trainer = SelfPlayTrainer(config)

print("Demarrage de l'entrainement...")
print(f"Checkpoints: {config.checkpoint_dir}")
print(f"Logs: {config.log_dir}")
print("-" * 50)

# ENTRAINEMENT
model = trainer.train()

print("\n" + "=" * 50)
print("ENTRAINEMENT TERMINE !")
print("=" * 50)

## 8. Evaluation finale

In [ ]:
print("Evaluation sur 100 parties...")
metrics = trainer.evaluate(n_games=100)

print(f"\nResultats finaux:")
print(f"  Taux de victoire: {metrics['win_rate']:.1%}")
print(f"  Reward moyen: {metrics['mean_reward']:.2f}")

## 9. Export ONNX pour le frontend

In [ ]:
from chato_rl.export import export_to_onnx

# Exporter le meilleur modele
best_model_path = config.checkpoint_dir / "best_model.zip"
final_model_path = config.checkpoint_dir / "final_model.zip"
onnx_path = config.checkpoint_dir / "extreme_ai.onnx"

model_path = best_model_path if best_model_path.exists() else final_model_path

print(f"Export depuis: {model_path}")
export_to_onnx(
    model_path=str(model_path),
    output_path=str(onnx_path),
    num_players=config.num_players,
)

# Taille du fichier
import os
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"\nModele exporte: {onnx_path}")
print(f"Taille: {size_mb:.1f} MB")

## 10. Telecharger le modele

In [ ]:
from google.colab import files

# Telecharger le modele ONNX
files.download(str(onnx_path))

print("\nLe fichier extreme_ai.onnx a ete telecharge.")
print("Place-le dans backend/models/extreme_ai/ pour l'utiliser.")

---

## Reprendre un entrainement interrompu

Si Colab s'est deconnecte, execute d'abord les cellules 1-5, puis cette cellule :

In [ ]:
from pathlib import Path
from chato_rl.training import SelfPlayTrainer
from chato_rl.training.config import A100Config

config = A100Config()

# Trouver le dernier checkpoint
checkpoints = sorted(config.checkpoint_dir.glob("checkpoint_*.zip"))
if checkpoints:
    last_checkpoint = checkpoints[-1]
    print(f"Dernier checkpoint trouve: {last_checkpoint}")
    
    # Extraire le timestep du nom
    timestep = int(last_checkpoint.stem.split("_")[1])
    remaining = config.total_timesteps - timestep
    print(f"Timesteps restants: {remaining:,}")
    
    # Reprendre
    trainer = SelfPlayTrainer(config)
    trainer.load_checkpoint(last_checkpoint)
    
    config.total_timesteps = remaining
    model = trainer.train()
else:
    print("Aucun checkpoint trouve. Lance un nouvel entrainement.")